# Wind Power Prediction Model Training
**Dataset:** Wind Power Generation Data - Forecasting  
**Link:** https://www.kaggle.com/datasets/mubashirrahim/wind-power-generation-data-forecasting  
**Output:** `wind_model.pkl`  
**Model:** XGBoost Regressor  
**Input features:** `wind_speed`, `wind_speed_50m`, `wind_direction`, `temperature_avg`  
**Target:** `wind_power_kw`

In [ ]:
import numpy as np
import pandas as pd
import joblib
import glob
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

## 1. Load Dataset

In [ ]:
files = glob.glob('/kaggle/input/**/*.csv', recursive=True)
print(files)
df = pd.read_csv(files[0])
print(df.shape)
df.head()

In [ ]:
print(df.columns.tolist())
print(df.dtypes)
print(df.isnull().sum())

## 2. Map Columns to Model Features
The dataset has wind speed at multiple heights (10m, 50m, hub height), wind direction, temperature, and power output.
We map them to the 4 features our `wind.py` service expects.

In [ ]:
# Auto-detect columns — adjust if needed after seeing column names above
def find_col(df, keywords):
    for kw in keywords:
        match = [c for c in df.columns if kw.lower() in c.lower()]
        if match:
            return match[0]
    return None

wind_10m_col   = find_col(df, ['wind_speed_10', 'ws_10', 'speed_10', 'wind_speed'])
wind_50m_col   = find_col(df, ['wind_speed_50', 'ws_50', 'speed_50', 'wind_50m'])
wind_dir_col   = find_col(df, ['wind_dir', 'direction', 'wd'])
temp_col       = find_col(df, ['temperature', 'temp', 'ambient'])
power_col      = find_col(df, ['power', 'active_power', 'lv_activepower', 'output'])

print(f'wind_speed (10m): {wind_10m_col}')
print(f'wind_speed_50m:   {wind_50m_col}')
print(f'wind_direction:   {wind_dir_col}')
print(f'temperature_avg:  {temp_col}')
print(f'power output:     {power_col}')

In [ ]:
df = df.rename(columns={
    wind_10m_col: 'wind_speed',
    wind_dir_col: 'wind_direction',
    power_col:    'wind_power_kw',
})

# wind_speed_50m: use dedicated column or estimate from 10m using power law
# V_50 = V_10 * (50/10)^0.143  (Hellmann exponent for open terrain)
if wind_50m_col:
    df = df.rename(columns={wind_50m_col: 'wind_speed_50m'})
else:
    df['wind_speed_50m'] = df['wind_speed'] * (50/10)**0.143

# temperature
if temp_col:
    df = df.rename(columns={temp_col: 'temperature_avg'})
else:
    df['temperature_avg'] = 20.0

df[['wind_speed', 'wind_speed_50m', 'wind_direction', 'temperature_avg', 'wind_power_kw']].describe()

## 3. Clean Data

In [ ]:
FEATURES = ['wind_speed', 'wind_speed_50m', 'wind_direction', 'temperature_avg']
TARGET = 'wind_power_kw'

df_clean = df[FEATURES + [TARGET]].dropna()
df_clean = df_clean[df_clean[TARGET] >= 0]  # remove negative power (sensor errors)

# Remove cut-in/cut-out outliers: turbines produce 0 below ~3 m/s and above ~25 m/s
df_clean = df_clean[df_clean['wind_speed'].between(0, 30)]

print(f'Clean rows: {len(df_clean)}')
print(f'Power range: {df_clean[TARGET].min():.1f} – {df_clean[TARGET].max():.1f} kW')

## 4. Train Model

In [ ]:
X = df_clean[FEATURES].values
y = df_clean[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=50)

y_pred = model.predict(X_test)
print(f'MAE: {mean_absolute_error(y_test, y_pred):.2f} kW')
print(f'R²:  {r2_score(y_test, y_pred):.4f}')

In [ ]:
for feat, imp in zip(FEATURES, model.feature_importances_):
    print(f'{feat}: {imp:.4f}')

## 5. Save Model

In [ ]:
joblib.dump(model, 'wind_model.pkl')
print('Saved: wind_model.pkl')

# Verify — wind_speed, wind_speed_50m, wind_direction, temperature_avg
loaded = joblib.load('wind_model.pkl')
test_input = np.array([[8.5, 10.2, 180.0, 22.0]])
pred = loaded.predict(test_input)[0]
print(f'Test prediction: {pred:.2f} kW')